# LangChain Agent with FastAPI - Structured Showcase
This notebook walks through building a cleanly structured REST API that integrates Large Language Models (LLMs) using LangChain and FastAPI. It follows a Domain-Driven Design (DDD) approach to separate concerns.

In [ ]:
# Run this cell to install the required dependencies
!pip install fastapi uvicorn langchain-openai pydantic-settings nest-asyncio

### 1. System Layer (`config.py`)
Manages application-wide configurations and environment variables securely.

In [ ]:
import os
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    app_name: str = "LangChain Agent API"
    
    # Please replace this with your actual OpenAI API key, or use a .env file
    openai_api_key: str = "sk-your-api-key-here" 
    
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8")

# Instantiate settings to be used across the app
settings = Settings()

### 2. Domain Layer (`schemas.py`)
Defines the exact shape of our data using Pydantic models. This ensures strict data validation.

In [ ]:
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    """What the user sends to our API."""
    query: str = Field(..., description="The question or prompt for the agent.")

class ChatResponse(BaseModel):
    """What our API returns to the user."""
    answer: str = Field(..., description="The response generated by the LangChain agent.")

### 3. Generator Layer (`langchain_agent.py`)
The "Brain". This encapsulates all LangChain and OpenAI logic, prompt templates, and LLM chains.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

class SimpleAgentGenerator:
    def __init__(self, api_key: str):
        # Initialize the LLM
        self.llm = ChatOpenAI(
            temperature=0.7, 
            openai_api_key=api_key,
            model="gpt-3.5-turbo"
        )
        
        # Define the system's persona and prompt
        self.prompt = PromptTemplate(
            input_variables=["query"],
            template="You are a highly intelligent and helpful tutor. Answer the student's query clearly and concisely.\n\nQuery: {query}"
        )
        
        # Create a simple chain
        self.chain = self.prompt | self.llm | StrOutputParser()

    def generate_response(self, query: str) -> str:
        try:
            return self.chain.invoke({"query": query})
        except Exception as e:
            return f"Error generating response: {str(e)}"

### 4. Service Layer (`agent_service.py`)
The "Manager". Connects the validated request to the AI Generator, handling any business logic.

In [ ]:
class AgentService:
    def __init__(self):
        # Instantiate the generator using system configurations
        self.generator = SimpleAgentGenerator(api_key=settings.openai_api_key)

    def process_chat(self, request: ChatRequest) -> ChatResponse:
        """Processes the request and coordinates with the generator."""
        user_query = request.query
        llm_answer = self.generator.generate_response(user_query)
        return ChatResponse(answer=llm_answer)

### 5. Controller Layer (`agent_controller.py`)
Strictly handles HTTP traffic. Receives the request, passes it to the Service, and returns the response.

In [ ]:
from fastapi import APIRouter, Depends, HTTPException

# Create a FastAPI router
router = APIRouter(prefix="/api/v1/agent", tags=["LangChain Agent"])

# Dependency injection for the service
def get_agent_service():
    return AgentService()

@router.post("/chat", response_model=ChatResponse)
async def chat_endpoint(
    request: ChatRequest, 
    service: AgentService = Depends(get_agent_service)
):
    if not request.query.strip():
        raise HTTPException(status_code=400, detail="Query cannot be empty.")
        
    return service.process_chat(request)

### 6. The Entry Point (`main.py`)
Initializes FastAPI and mounts the controllers.

In [ ]:
from fastapi import FastAPI

# Initialize FastAPI app
app = FastAPI(
    title=settings.app_name,
    description="A cleanly structured API demonstrating LangChain integration.",
    version="1.0.0"
)

# Include the controllers (routers)
app.include_router(router)

@app.get("/", tags=["Health Check"])
def root():
    return {"status": "online", "message": f"Welcome to {settings.app_name}"}

### 7. Run the FastAPI Server in Jupyter
Because Jupyter Notebooks already run in an asynchronous event loop, we need to use `nest_asyncio` to allow Uvicorn to run properly inside a cell.

In [ ]:
import nest_asyncio
import uvicorn

# Allow nested event loops
nest_asyncio.apply()

print("Server starting... ")
print("==========================================================")
print("✅ GO TO: http://127.0.0.1:8000/docs to test your agent! ")
print("==========================================================\n")

# Run the server
uvicorn.run(app, host="127.0.0.1", port=8000)